# 06 · Freeze alarm thresholds and primary policy

Uses only model-validation data to freeze alarm thresholds and predeclares q90 as the primary v1 policy.

**Checkpoint date:** 2026-10-05.

In [ ]:
#!/usr/bin/env python3
"""
Freeze binary alarm thresholds and designate the primary Gray-Box policy
BEFORE opening policy_validation outcomes.

Uses ONLY:
- model_validation (3,905 cases) for alarm threshold selection
- already-frozen probability calibration (raw/raw/raw)
- already-frozen policy candidate family

No policy_validation labels are read.
No later roles are read.
No model fitting/retraining.
No GPU use.
"""

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json
import numpy as np
import pandas as pd

MASTER = Path("/home/abmoses2000/graybox_aia72_master_predictions_v2_20261005/graybox_aia72_master_predictions.csv.gz")
MASTER_SHA = "17300093b88b7c61ecadf5eb28acba0d6d023b340d7740807852dd4352b22b46"

SHARP = Path("/home/abmoses2000/sharp72_predictions_frozen_20261002.csv.gz")
SHARP_SHA = "7368c002a978321cdc67221564f3b7901687badc7aba28ab9df615179d1e017b"

CAL_SELECTION = Path("/home/abmoses2000/multimodal72_calibration_v1_20261005T082136654618Z/selection.json")
POLICY_DIR = Path("/home/abmoses2000/graybox_aia72_policy_freeze_v1_20261005")
POLICY_FAMILY = POLICY_DIR / "policy_family.json"
POLICY_PROTOCOL = POLICY_DIR / "protocol.json"

OUT = Path("/home/abmoses2000/graybox_aia72_alarm_and_primary_policy_freeze_v1_20261005")
if OUT.exists():
    raise SystemExit(f"Refusing to overwrite existing output: {OUT}")
OUT.mkdir(parents=True)

EXPECTED_ROWS = 3905
SEEDS = [17,29,43]

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(4*1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def require(cond, msg):
    if not cond:
        raise ValueError(msg)

def confusion(y, pred):
    y = np.asarray(y, dtype=int)
    pred = np.asarray(pred, dtype=int)
    tp = int(((y==1)&(pred==1)).sum())
    tn = int(((y==0)&(pred==0)).sum())
    fp = int(((y==0)&(pred==1)).sum())
    fn = int(((y==1)&(pred==0)).sum())
    return tp, fp, fn, tn

def metrics(y, p, t):
    pred = (p >= t).astype(int)
    tp, fp, fn, tn = confusion(y, pred)
    tpr = tp/(tp+fn) if (tp+fn) else np.nan
    tnr = tn/(tn+fp) if (tn+fp) else np.nan
    tss = tpr + tnr - 1
    denom = ((tp+fn)*(fn+tn) + (tp+fp)*(fp+tn))
    hss = (2*(tp*tn - fn*fp)/denom) if denom else np.nan
    precision = tp/(tp+fp) if (tp+fp) else np.nan
    f1 = (2*precision*tpr/(precision+tpr)) if precision==precision and tpr==tpr and (precision+tpr)>0 else np.nan
    return {
        "threshold": float(t),
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
        "recall": float(tpr), "specificity": float(tnr),
        "tss": float(tss), "hss": float(hss),
        "precision": float(precision) if precision==precision else None,
        "f1": float(f1) if f1==f1 else None,
    }

def choose_threshold(y, p):
    # Candidate thresholds are all distinct observed probabilities plus 0 and 1.
    candidates = np.unique(np.concatenate(([0.0], p, [1.0])))
    rows = [metrics(y, p, t) for t in candidates]
    df = pd.DataFrame(rows)
    # Frozen deterministic rule:
    # 1) max TSS
    # 2) max HSS
    # 3) max recall
    # 4) lower threshold
    chosen = df.sort_values(
        ["tss","hss","recall","threshold"],
        ascending=[False,False,False,True],
        kind="mergesort"
    ).iloc[0].to_dict()
    return chosen, df

for p in [MASTER, SHARP, CAL_SELECTION, POLICY_FAMILY, POLICY_PROTOCOL]:
    require(p.is_file(), f"Missing input: {p}")
require(sha256(MASTER)==MASTER_SHA, "Master SHA changed")
require(sha256(SHARP)==SHARP_SHA, "SHARP SHA changed")

selection = json.loads(CAL_SELECTION.read_text())
require(selection["selected_methods"] == {"sharp":"raw","aia":"raw","sharp_aia":"raw"},
        "Probability calibration no longer raw/raw/raw")

policy_family = json.loads(POLICY_FAMILY.read_text())
policy_protocol = json.loads(POLICY_PROTOCOL.read_text())
require(policy_family["status"]=="POLICY_CANDIDATE_FAMILY_FROZEN", "Policy family not frozen")
require(policy_protocol["policy_validation_outcomes_read"] is False, "Policy validation already read")
require(policy_protocol["later_outcomes_read"] is False, "Later outcomes already read")

master = pd.read_csv(MASTER, low_memory=False)
sharp = pd.read_csv(SHARP, low_memory=False)

a = master.loc[master["role"].astype(str)=="model_validation"].copy()
s = sharp.loc[sharp["role"].astype(str)=="model_validation"].copy()
require(len(a)==EXPECTED_ROWS and len(s)==EXPECTED_ROWS, "model_validation support changed")

a["forecast_case_id"]=a["forecast_case_id"].astype(str)
s["forecast_case_id"]=s["forecast_case_id"].astype(str)
require(a["forecast_case_id"].tolist()==s["forecast_case_id"].tolist(), "SHARP/AIA model_validation IDs/order mismatch")

y = pd.to_numeric(a["label"], errors="raise").astype(int).to_numpy()
ys = pd.to_numeric(s["label"], errors="raise").astype(int).to_numpy()
require(np.array_equal(y,ys), "SHARP/AIA model_validation labels disagree")
require(set(np.unique(y)).issubset({0,1}), "Non-binary labels")

aia_p = pd.to_numeric(a["probability"], errors="raise").to_numpy(float)
sharp_p = pd.to_numeric(s["probability_gru_mean"], errors="raise").to_numpy(float)
fusion_p = 0.5*sharp_p + 0.5*aia_p

branches = {"sharp":sharp_p, "aia":aia_p, "sharp_aia":fusion_p}
chosen = {}
curve_files = {}

for branch,p in branches.items():
    require(np.isfinite(p).all() and ((p>=0)&(p<=1)).all(), f"Bad probabilities: {branch}")
    best, curve = choose_threshold(y,p)
    chosen[branch]=best
    path = OUT / f"{branch}_threshold_curve.csv"
    curve.to_csv(path,index=False)
    curve_files[path.name]=sha256(path)

threshold_table = pd.DataFrame([
    {"branch":b, **m} for b,m in chosen.items()
])
threshold_path = OUT / "alarm_thresholds.csv"
threshold_table.to_csv(threshold_path,index=False)

# Freeze the middle candidate q90 as primary before policy-validation outcomes.
# q80 = stricter sensitivity; q95 = looser sensitivity.
candidate_ids = [x["policy_id"] for x in policy_family["candidates"]]
require(candidate_ids == ["policy_q80","policy_q90","policy_q95"], f"Unexpected policy candidates: {candidate_ids}")

primary_policy = {
    "status": "PRIMARY_POLICY_PREDECLARED_BEFORE_POLICY_VALIDATION",
    "primary_policy_id": "policy_q90",
    "sensitivity_policy_ids": ["policy_q80","policy_q95"],
    "rationale": (
        "The middle pre-frozen uncertainty cut-off is designated primary before policy-validation outcomes are opened. "
        "q80 is the stricter sensitivity analysis and q95 the looser sensitivity analysis. "
        "Policy-validation will evaluate, not re-select, the primary policy."
    ),
    "selection_on_policy_validation_allowed": False,
    "policy_validation_outcomes_read": False,
    "scientific_acceptance": False,
}

primary_path = OUT / "primary_policy_declaration.json"
primary_path.write_text(json.dumps(primary_policy,indent=2)+"\n")

protocol = {
    "status": "ALARM_THRESHOLDS_AND_PRIMARY_POLICY_FROZEN",
    "scientific_acceptance": False,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "horizon_hours": 72,
    "threshold_source_role": "model_validation",
    "threshold_source_rows": EXPECTED_ROWS,
    "threshold_selection_rule": "max TSS; tie max HSS; tie max recall; tie lower threshold",
    "probability_calibration": {"sharp":"raw_identity","aia":"raw_identity","sharp_aia":"raw_identity"},
    "alarm_thresholds": {b: float(m["threshold"]) for b,m in chosen.items()},
    "primary_policy_id": "policy_q90",
    "sensitivity_policy_ids": ["policy_q80","policy_q95"],
    "policy_validation_outcomes_read": False,
    "later_outcomes_read": False,
    "source_sha256": {
        "master_predictions": MASTER_SHA,
        "sharp_predictions": SHARP_SHA,
        "calibration_selection": sha256(CAL_SELECTION),
        "policy_family": sha256(POLICY_FAMILY),
        "policy_protocol": sha256(POLICY_PROTOCOL),
    },
    "output_sha256": {
        threshold_path.name: sha256(threshold_path),
        primary_path.name: sha256(primary_path),
        **curve_files,
    },
}

protocol_path = OUT / "protocol.json"
protocol_path.write_text(json.dumps(protocol,indent=2)+"\n")

print("===== ALARM + PRIMARY POLICY FREEZE COMPLETE =====")
print("model_validation rows:", EXPECTED_ROWS)
print("positives:", int(y.sum()))
print("policy-validation outcomes read: False")
print("\nFrozen alarm thresholds and validation metrics:")
print(threshold_table.to_string(index=False))
print("\nPrimary policy:", primary_policy["primary_policy_id"])
print("Sensitivity policies:", primary_policy["sensitivity_policy_ids"])
print("\nProtocol:")
print(json.dumps(protocol,indent=2))
print("\nSaved:", protocol_path)
print("Saved:", threshold_path)
print("Saved:", primary_path)
